In [4]:
cd /mnt/home/al2644/research/projects/perturb-r

/mnt/home/al2644/research/projects/perturb-r


In [5]:
import random
from fractions import Fraction
from typing import List, Tuple
import pandas as pd
from datasets import Dataset

import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

from utils.corrupt_num import *
from utils.chunk_r import *

# Check Eval Dataset

In [22]:
pwd

'/mnt/home/al2644/research/projects/perturb-r'

In [28]:
path = "./data/deepmath/"
dataset = load_from_disk(path)['test']
# eval_df = dataset.to_pandas()

In [43]:
file = '/mnt/home/al2644/research/projects/perturb-r/data/aime2425'
aime = load_from_disk(file)

In [47]:
aime['test'][0]

{'problem': 'Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: \n\\[\\log_2\\left({x \\over yz}\\right) = {1 \\over 2}\\]\n\\[\\log_2\\left({y \\over xz}\\right) = {1 \\over 3}\\]\n\\[\\log_2\\left({z \\over xy}\\right) = {1 \\over 4}\\]\nThen the value of $\\left|\\log_2(x^4y^3z^2)\\right|$ is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.',
 'solution': '33',
 'source': 'aime 24'}

# Reasoning Benchmark Eval

In [39]:
root = "./results/deepmath/benchmark/"
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
for fname in os.listdir(root):
    if 'correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['correct'].mean()}")

DeepMath-1.5B.pickle
Accuracy: 0.7511538461538462
Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-1epoch.pickle
Accuracy: 0.13846153846153847
Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-2epoch.pickle
Accuracy: 0.15115384615384617
Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-3epoch.pickle
Accuracy: 0.17038461538461538
Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-4epoch.pickle
Accuracy: 0.18
Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-5epoch.pickle
Accuracy: 0.18153846153846154
R1-Distill-Qwen-1.5B.pickle
Accuracy: 0.5969230769230769


In [34]:
def groupby_level (df):
    stats = df.groupby('difficulty')['correct'].mean()
    return stats

In [40]:
df = pd.read_pickle(os.path.join(root, "R1-Distill-Qwen-1.5B.pickle"))
groupby_level(df)

difficulty
3.0    0.800
3.5    0.585
4.0    0.710
4.5    0.660
5.0    0.705
5.5    0.635
6.0    0.650
6.5    0.545
7.0    0.575
7.5    0.495
8.0    0.475
8.5    0.435
9.0    0.490
Name: correct, dtype: float64

In [37]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-4epoch.pickle"))
groupby_level(df)

difficulty
3.0    0.415
3.5    0.155
4.0    0.230
4.5    0.185
5.0    0.155
5.5    0.115
6.0    0.105
6.5    0.155
7.0    0.150
7.5    0.215
8.0    0.155
8.5    0.130
9.0    0.175
Name: correct, dtype: float64

In [38]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-1.5B-DeepMath-level1-4-14k-sft-stage0-5epoch.pickle"))
groupby_level(df)

difficulty
3.0    0.380
3.5    0.155
4.0    0.310
4.5    0.185
5.0    0.140
5.5    0.095
6.0    0.120
6.5    0.130
7.0    0.150
7.5    0.155
8.0    0.195
8.5    0.150
9.0    0.195
Name: correct, dtype: float64

# Corrupted Numbers

In [840]:
pwd

'/home/al2644/research/codebase/reasoning/perturb-r'

In [841]:
root = "./results/countdown/corrupt_numbers/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    if 'still_correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['still_correct'].mean()}")

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.5202702702702703
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.5934684684684685


In [842]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))


In [844]:
df.groupby("start_pos")["still_correct"].mean()

start_pos
0.00    0.184685
0.25    0.720721
0.50    0.752252
0.75    0.716216
Name: still_correct, dtype: float64

In [854]:
df.groupby("start_pos")["if_answer"].mean()

start_pos
0.00    0.409910
0.25    0.765766
0.50    0.788288
0.75    0.995495
Name: if_answer, dtype: float64

In [858]:
example = df[(df["start_pos"] == 0.5) & (df["still_correct"] == 0.) & (df["if_answer"] == True)].iloc[0]

In [859]:
print(example["prompt"])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [13, 10, 27, 92, 31], create an equation that equals 742. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>


Okay, let's see. I need to use the numbers 13, 10, 27, 92, and 31 exactly once each, with basic operations to make 742. Hmm. Let me think. First, maybe I should try combining some numbers to get closer to 742. Since 742 is a relatively large number, maybe multiplication is involved here. Let me check the numbers: 92 is


the largest, then 31, 27, 10, 13.


Let me think about possible multiplications. If I multiply 92 by something, maybe? Let's see. 92 * 8 is 736, which is close to 742. But how do I get 8 from t

In [860]:
print(example["post_corruption_response"])

 

Alternatively, think of 92 * 10 = 920. Then 920 - 27 = 893. 893 - 13 = 880. 880 + 31 = 911. No, that's not helpful. 

Alternatively, 92 * 27 = 2484. Then 2484 - 31 = 2453. 2453 - 13 = 2440. 2440 + 10 = 2450. Not 742. 

Alternatively, 92 * 13 = 1204. Then 1204 - 31 = 1173. 1173 - 27 = 1146. 1146 - 10 = 1136. That's way over. 

Alternatively, 92 * 31 = 2852. Then 2852 - 27 = 2825. 2825 - 13 = 2812. 2812 - 10 = 2802. No. 

Alternatively, 31 * 10 = 310. Then 310 + 27 = 337. 337 + 92 = 429. 429 + 13 = 442. No. 

Wait, maybe 27 * 13 = 351. Then 351 * 2 = 702. Then 702 + 40 = 742. How to get 40 from 92 and 31. 92 - 31 = 61. Not 40. 92 - 31 is 61, then 61 - 10 = 51. No. 

Alternatively, 27 * 10 = 270. Then 270 + 92 = 362. 362 * something? 362 * 2 is 724. Then 724 + 18 = 742. How to get 18 from 13 and 31. 31 - 13 = 18. Oh! Wait! That's possible. 

So let's see: (27 * 10) + 92 = 362. Then 362 + (31 - 13) = 362 + 18 = 380. No, that's not. Wait, but if I do 27 * 10 + 92 + (31 -13) = 362 + 92 + 

In [695]:
wrong_answer[['level', 'type']].value_counts()

level  type         
5      post_</think>    80
6      post_</think>    21
5      pre_</think>     17
6      pre_</think>      5
Name: count, dtype: int64

# Distractor Injection

In [796]:
root = "./results/countdown/inject_distractor/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    print(fname)
    df = pd.read_pickle(os.path.join(root, fname))
    print(f"Original Accuracy: {df['original_correct'].mean()}")
    print(f"Distractor Accuracy: {df['distractor_correct'].mean()}")

df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Original Accuracy: 0.3626237623762376
Distractor Accuracy: 0.5006188118811881
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Original Accuracy: 0.46163366336633666
Distractor Accuracy: 0.47277227722772275


In [797]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.047030,0.816832
0.25,0.544554,0.292079
0.50,0.462871,0.400990
0.75,0.396040,0.492574


In [798]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.009901,0.938119
0.25,0.670792,0.252475
0.50,0.613861,0.309406
0.75,0.551980,0.391089


In [799]:
df

,nums,target,example_solution,problem,level,task,original_response,reasoning_chunks,prefix_ratio,distractor_ratio,distract_nums,distract_target,distract_reasoning,prompt,post_distraction_response,original_correct,distractor_correct
0,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.25,"[86, 91, 87, 68, 77, 66]",165,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,"Not sure. Alternatively, 87 - 66 = 21. 91 - 7...",0.0,1.0
1,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.5,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,33 + 85 is 118. 118 - 82 is 36. 36 +73 is 109...,0.0,1.0
2,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.75,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,So the answer would be that expression. So th...,0.0,1.0
3,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,1.0,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,The answer should be in <answer> tags. So <an...,0.0,1.0
4,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.25,0.25,"[25, 87, 42, 34, 57]",63,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"87 - 57 is 30, then 30 - 25 is 5, not helpful....",1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1611,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.5,1.0,"[75, 96, 96, 56, 68]",129,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"Therefore, the equation is 96 - 75 + 96 + 68 ...",1.0,0.0
1612,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.25,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,Maybe (64 - 51) = 13. Then 13 + 48 = 61. Then...,1.0,0.0
1613,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.5,"[71, 93, 76, 10, 96]",184,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,204 is not 184. What if I do 76 + 93 = 169. 16...,0.0,0.0
1614,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.75,"[17, 76, 59, 23, 18]",41,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan

In [802]:
idx = 7
example = df.iloc[idx]
prefix_ratio, distractor_ratio = example["prefix_ratio"], example["distractor_ratio"]
print("prefix_ratio: ", prefix_ratio)
print("distractor_ratio: ", distractor_ratio)
prompt, response, original_correct, distractor_correct = example["prompt"], example["post_distraction_response"], example["original_correct"], example["distractor_correct"]

print("original correct?: ", original_correct)
print("distract correct?: ", distractor_correct)

prefix_ratio:  0.25
distractor_ratio:  1.0
original correct?:  0.0
distract correct?:  1.0


In [803]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [50, 77, 64, 67, 73], create an equation that equals 331. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to use each of the numbers 50, 77, 64, 67, and 73 exactly once with basic operations to make 331. Hmm. Let me think about possible combinations.Wait. Let me think again: Okay, let's see. I need to use the numbers 86, 91, 87, 68, 77, and 66 exactly once each, with basic arithmetic operations to make 165. Hmm, first, maybe I should think about possible combinations. Let me list the numbers again: 86, 91, 87, 68, 77, 66. All have to be used once. The target is 165.What's 165? Let me think abo

In [804]:
print(response)

 Which is the same as the previous grouping. Alternatively, written as (66 + 77) + (86 - 68) + (91 - 87). Either way, the operations are all allowed. Let me check again if the order is correct. Let me compute the expression step by step. 66 + 77 is 143. Then 143 + 86 is 229. Then subtract 68: 229 - 68 = 161. Then add 91: 161 + 91 = 252. Then subtract 87: 252 - 87 = 165. Correct. So that's the right equation.
</think>

<answer> 66 + 77 + 86 - 68 + 91 - 87 </answer>



In [ ]:
evals
    0. same set of stress tests for both math and countdown
    1. if observations on countdown and math are consistent
    2. if the corruption happens in the first chunk
    3. check the eval results for sft 3 epoch

In [ ]:
training todo:
    
    mix original question and corrupted prefix
    1. collect offline rl prefix ==> rl (save checkpoints for later eval)
    2. rl on same flops ==> same number of problems (assuming problems i.i.d)
    3. 1st rl collect for rejection sampling
    
    
generalization:
    1. different types of corruption (distractor injection, just wrong answer at the end)
    2. different degrees of digit corruption
    3. different task complexity
    


In [ ]:
generate more level 5, 6, 7 questions 50K each
use r1 to generate corrupted prefix + collect recovery + not recover (50 50)

we can launch 2 first

# Check test set

In [42]:
df1 = pd.read_parquet("../rlvr/data/train/countdown_level5_random_thought_30k/test.parquet")
df2 = pd.read_parquet("../rlvr/data/train/countdown_level5_random_thought_only_15k/test.parquet")

In [43]:
df1

,data_source,prompt,ability,reward_model,extra_info
0,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [50, 77, 64, 67, ...","{'index': 0, 'level': '6', 'split': 'test'}"
1,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [88, 84, 11, 71, ...","{'index': 1, 'level': '6', 'split': 'test'}"
2,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [35, 48, 94, 46, ...","{'index': 2, 'level': '6', 'split': 'test'}"
3,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [18, 58, 64, 42, ...","{'index': 3, 'level': '6', 'split': 'test'}"
4,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [37, 18, 51, 39, ...","{'index': 4, 'level': '6', 'split': 'test'}"
...,...,...,...,...,...
295,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [95, 38, 13, 72, ...","{'index': 295, 'level': '7', 'split': 'test'}"
296,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [51, 100, 99, 91,...","{'index': 296, 'level': '7', 'split': 'test'}"
297,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [44, 2, 3, 41, 85...","{'index': 297, 'level': '7', 'split': 'test'}"
298,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [55, 20, 64, 3, 4...","{'index': 298, 'level': '7', 'split': 'test'}"


In [44]:
df2

,data_source,prompt,ability,reward_model,extra_info
0,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [50, 77, 64, 67, ...","{'index': 0, 'level': '6', 'split': 'test'}"
1,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [88, 84, 11, 71, ...","{'index': 1, 'level': '6', 'split': 'test'}"
2,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [35, 48, 94, 46, ...","{'index': 2, 'level': '6', 'split': 'test'}"
3,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [18, 58, 64, 42, ...","{'index': 3, 'level': '6', 'split': 'test'}"
4,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [37, 18, 51, 39, ...","{'index': 4, 'level': '6', 'split': 'test'}"
...,...,...,...,...,...
295,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [95, 38, 13, 72, ...","{'index': 295, 'level': '7', 'split': 'test'}"
296,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [51, 100, 99, 91,...","{'index': 296, 'level': '7', 'split': 'test'}"
297,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [44, 2, 3, 41, 85...","{'index': 297, 'level': '7', 'split': 'test'}"
298,countdown,[{'content': 'Please reason and answer the fol...,math,"{'ground_truth': {'numbers': [55, 20, 64, 3, 4...","{'index': 298, 'level': '7', 'split': 'test'}"
